# U-02: Quantile-ML + Conformal Uncertainty for the Recursive-Rollout Models

**Objective:** Given B-10/B-13's point forecasts are the best available (B-14/B-15 closed the
tuning side-thread), quantify their uncertainty responsibly -- both a quantile-regression-style
interval (adapts to known heteroscedasticity: spikes vs quiet periods, tower-to-tower differences)
and a conformal-calibrated version (guarantees the interval's coverage is actually correct against
held-out data, not just well-intentioned). Fresh methodology -- **not** based on the old
`U01_uncertainty.ipynb` (different harness, not used as precedent here); `pinball`/`picp`/`mpiw`
are freshly written in `src/evaluation/metrics.py`, not migrated from U-01's inline versions.

**Per-model quantile mechanism:**
- **RF**: quantile-regression-forest trick on the *already-fitted* point model (no retraining) --
  quantiles from the empirical distribution of the forest's individual tree predictions.
- **XGB/LightGBM**: 3 separately-fit quantile-objective models per anchor (q=0.05/0.5/0.95), same
  hyperparameters as B-10's point models otherwise (no new HPO).
- **SARIMAX**: `get_forecast().conf_int()` -- quantiles essentially for free from the existing fit.
- **TFT**: quantile head deferred (a real architecture change, out of this experiment's bounded
  scope) -- conformal-wraps its own point-forecast chain instead.
- **TabPFN**: native quantile output via `tabpfn-time-series`'s own `quantiles=` parameter --
  genuine library support (confirmed via its signature), no fallback needed.
- **Ensembles**: no single native mechanism -- median = mean of the 4 constituents' medians (same
  as B-10's own point-forecast ensemble definition); raw interval = mean of constituent q0.05/q0.95
  bounds. Conformal-calibrated on top, same as everything else.

**Conformal calibration**: leave-one-anchor-out, per lead-time bin (reusing `bin_metrics`'s 6 bins,
necessary given the rollout's well-documented heteroscedasticity by lead time). For each anchor as
the held-out test anchor, calibrate using pooled absolute residuals from the *other 4* anchors,
split by bin. RAW (pre-calibration) and CALIBRATED (post-calibration) coverage/width/pinball are
both reported, so the value calibration actually adds is visible -- not asserted, measured.

This notebook demonstrates the full pipeline on a **bounded 2-anchor example** (2019, 2021; Tower
4) -- enough to exercise the leave-one-anchor-out calibration logic (needs >= 2 anchors) without
the full sweep's ~20+ minute runtime. The full 5-anchor x 3-tower sweep runs as
`u02_multi_anchor_tower.py`; fan-chart visualizations (actual/gap-filled/median + shaded
conformal band) are generated by `u02_fanchart_plots.py` into `results/figures/u02_fancharts/`.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

import models.recursive_rollout as rr
from evaluation.metrics import pinball, picp, mpiw

HOURLY = "../../data/Hourly"
RESULTS = "../../results"
N_DAYS = 365
TOWERS = [2, 4, 9]
TOWER = 4
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
QUANTILES = (0.05, 0.5, 0.95)
ALPHA = 0.10

dv = pd.read_csv(f"{HOURLY}/forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}
feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
print(f"Tower {TOWER}, {len(feat_cols)} features, quantiles={QUANTILES}")

Tower 4, 44 features, quantiles=(0.05, 0.5, 0.95)


## RF quantile: the quantile-regression-forest trick (no retraining)

In [2]:
ANCHOR = pd.Timestamp("2021-12-16")
target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

pool = []
for t in TOWERS:
    df = T[t].copy(); df["target"] = df["y_gapfilled"]
    for d in DUM:
        df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= ANCHOR])
tr = pd.concat(pool); tr = tr[tr["target"].notna()]

imp_ = SimpleImputer(strategy="mean")
Xi = imp_.fit_transform(tr[feat_cols].values)
rf = RandomForestRegressor(n_estimators=500, max_features=0.5, min_samples_leaf=10, n_jobs=-1, random_state=42).fit(Xi, tr["target"].values)
rf_adapter = rr.RFQuantileAdapter(rf)

dft = T[TOWER]
history_init = dft.loc[:ANCHOR, "y_gapfilled"].copy()
fx_frame = dft.loc[target_dates, FX_B + ["ar_fc_dlag1"]].copy()
fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0

df_rf = rr.tree_rollout_quantile(rf_adapter, imp_, feat_cols, fx_frame, history_init, ANCHOR, N_DAYS, QUANTILES)
print(df_rf.head())
print(f"\nMedian tracks point forecast: any q05 > median? {(df_rf[0.05] > df_rf['median']).any()} (should be False)")

                0.05        0.5       0.95     median
2021-12-17  5.388333   9.304010  15.847676   9.304010
2021-12-18  5.537784   9.309417  14.096126   9.309417
2021-12-19  5.388333   9.153929  14.045452   9.153929
2021-12-20  5.590073   9.127926  13.648976   9.127926
2021-12-21  5.628152  10.626465  22.282721  10.626465

Median tracks point forecast: any q05 > median? False (should be False)


## XGB quantile: 3 separately-fit quantile-objective models

In [3]:
xgb_models = {}
for q in QUANTILES:
    m = XGBRegressor(n_estimators=400, max_depth=2, learning_rate=0.02, min_child_weight=10,
                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                      objective="reg:quantileerror", quantile_alpha=q)
    m.fit(Xi, tr["target"].values)
    xgb_models[q] = m
xgb_adapter = rr.MultiModelQuantileAdapter(xgb_models)
df_xgb = rr.tree_rollout_quantile(xgb_adapter, imp_, feat_cols, fx_frame, history_init, ANCHOR, N_DAYS, QUANTILES)
print(df_xgb.head())

                0.05        0.5       0.95     median
2021-12-17  4.947432  11.095135  21.827591  11.095135
2021-12-18  4.947432  11.065365  21.209393  11.065365
2021-12-19  4.844361  10.977723  20.587151  10.977723
2021-12-20  4.844361  10.716429  19.803598  10.716429
2021-12-21  3.412994  10.905765  28.326189  10.905765


## SARIMAX quantile: `get_forecast().conf_int()` -- essentially free

In [4]:
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean", "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
y = dft["y_gapfilled"].astype(float)
X = dft[EXOG_B].astype(float).ffill().bfill()
best = None
for p in [1, 2, 3]:
    for q in [0, 1, 2]:
        try:
            m = SARIMAX(y.loc[:ANCHOR], exog=X.loc[:ANCHOR], order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]:
                best = (res.aic, (p, 1, q), res)
        except Exception:
            pass
sarimax_res = best[2]
df_sarimax = rr.sarimax_quantile(sarimax_res, target_dates, X.loc[target_dates], alpha=ALPHA)
print(df_sarimax.head())

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


               median       0.05       0.95
2021-12-17  12.838300 -34.664832  60.341431
2021-12-18  11.321776 -38.345661  60.989214
2021-12-19  14.299401 -36.781008  65.379811
2021-12-20  12.163115 -39.852476  64.178706
2021-12-21  17.021528 -35.618445  69.661500


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Conformal calibration + raw-vs-calibrated evaluation (script extension)

The full pipeline (all 8 models, leave-one-anchor-out conformal calibration across the 5-anchor
sweep, all 3 towers) runs as `u02_multi_anchor_tower.py`, producing `results/u02_chains.csv`
(per-day quantile/point predictions) and `results/u02_summary.csv` (raw vs conformal-calibrated
PICP/MPIW/pinball per model/tower/bin). Fan-chart visualizations: `u02_fanchart_plots.py` ->
`results/figures/u02_fancharts/`.

See `U02_results.md` for the full narrative and the "what does conformal calibration actually buy
you here" comparison, backed by real numbers.